In [32]:
# 1. Import required modules
# StateGraph, START, END: LangGraph core components for building workflows.
# TypedDict, NotRequired: For defining the state schema type hints.
# ChatGoogleGenerativeAI: LangChain wrapper for Google's Gemini LLM.
# load_dotenv: Loads environment variables (like GOOGLE_API_KEY) from a .env file.
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, NotRequired
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()


True

In [33]:
# 2. Initialize the LLM
# We create an instance of the Gemini 3.1 Flash model.
# temperature=0.7 allows for a slight bit of creativity in the response.
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0.7)

In [34]:
# 3. Define the Graph State (Shared Memory)
# The state schema defines the data that flows through the graph.
# input: The user's question (required to start).
# output: The LLM's answer (NotRequired because it's generated by the node).
class PromptChainState(TypedDict):
    topic: str
    outline: NotRequired[str]
    blog: NotRequired[str]
    summary: NotRequired[str]


In [35]:
# 4. Define the Nodes (Workers/Actions)
# Each node performs a specific step in the sequential chain.
# They take the current state, call the LLM, and return a dictionary
# with the specific key they want to update in the global state.

def generate_outline(state: PromptChainState) -> dict:
    """Generates an outline based on the topic."""
    prompt = f"Generate an outline for a blog post about {state['topic']}. in just 3 points"
    response = llm.invoke(prompt)
    content = response.content[0]['text'] if isinstance(response.content, list) else response.content
    return {"outline": content}

def generate_blog(state: PromptChainState) -> dict:
    """Writes a blog post based on the generated outline."""
    outline = state.get("outline")
    if outline is None:
        raise ValueError("Missing 'outline' in state before generating blog.")
    prompt = f"Write a blog post based on this outline:\n{outline} in 3 paragraphs."
    response = llm.invoke(prompt)
    content = response.content[0]['text'] if isinstance(response.content, list) else response.content
    return {"blog": content}

def generate_summary(state: PromptChainState) -> dict:
    """Summarizes the blog post."""
    blog = state.get("blog")
    if blog is None:
        raise ValueError("Missing 'blog' in state before generating summary.")
    prompt = f"Summarize this blog post in 3 sentences:\n{blog} in 3 sentences."
    response = llm.invoke(prompt)
    content = response.content[0]['text'] if isinstance(response.content, list) else response.content
    return {"summary": content}

In [36]:
# 5. Build and Compile the Graph
# Initialize the graph with our PromptChainState schema.
graph = StateGraph(PromptChainState)

# Register all three nodes
graph.add_node("outline", generate_outline)
graph.add_node("blog", generate_blog)
graph.add_node("summary", generate_summary)

# Define the sequential execution flow (edges)
graph.add_edge(START, "outline")    # Entry point -> outline
graph.add_edge("outline", "blog")   # outline -> blog
graph.add_edge("blog", "summary") # blog -> summary
graph.add_edge("summary", END)      # summary -> exit point

# Compile the graph into a runnable workflow object
workflow = graph.compile()

In [37]:
# 6. Execute the workflow
# Invoke the graph with the initial state containing the user's topic.
# IMPORTANT NOTE: If you hit a "RESOURCE_EXHAUSTED" (429) error,
# it means you've exceeded the Gemini Free Tier rate limit.
# Simply wait a minute or two and run this cell again!

result = workflow.invoke({"topic": "The Future of AI in Healthcare"})

print("--- OUTLINE ---")
print(result["outline"])
print("\n--- BLOG POST ---")
print(result["blog"])
print("\n--- SUMMARY ---")
print(result["summary"])

--- OUTLINE ---
Here is a concise outline for a blog post on the future of AI in healthcare:

1.  **Precision Medicine and Diagnostics:** Exploring how AI algorithms analyze vast datasets, medical imaging, and genetic profiles to provide earlier, more accurate disease detection and personalized treatment plans.
2.  **Operational Efficiency and Administrative Automation:** Discussing the role of AI in streamlining hospital workflows, automating clinical documentation, and reducing burnout for healthcare professionals.
3.  **Ethical Considerations and Human-Centric Care:** Addressing the challenges of data privacy, algorithmic bias, and the necessity of maintaining the "human touch" as AI becomes an integral part of the patient-provider relationship.

--- BLOG POST ---
The integration of artificial intelligence into healthcare is ushering in a new era of precision medicine, fundamentally shifting how we approach diagnostics and treatment. By leveraging sophisticated algorithms to analyze